In [1]:
import pandas as pd
import numpy as np
import re
import os

In [2]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

### AI adoption rate - 2023

The dependent variable is the proportion of firms within each country - sector using at least one type of AI in 2023. Unit of measurement - share of the firms. PC_ENT

In [ ]:
ai_adopt = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/ain2.csv', index_col=0)
print(ai_adopt)

       freq size_emp nace_r2           indic_is    unit geo  num_2021  \
0         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  AT       NaN   
1         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BA       NaN   
2         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BE       NaN   
3         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BG       NaN   
4         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  CY       NaN   
...     ...      ...     ...                ...     ...  ..       ...   
221987    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RO       NaN   
221988    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RS       NaN   
221989    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SE       NaN   
221990    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SI       NaN   
221991    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SK       NaN   

       flag_2021  num_2023 flag_2023  num_2024 flag_2024  
0            NaN      8.24       NaN       NaN       NaN  
1    

In [80]:
un = pd.unique(ai_adopt['size_emp'])
print(un)

['GE10']


In [81]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt_2023 = ai_adopt.copy()
ai_adopt_2023 = ai_adopt.drop(columns=[ 'freq', 'size_emp', 'num_2021', 'flag_2021', 'flag_2023', 'num_2024', 'flag_2024'])

ai_adopt_2023 = ai_adopt_2023[
    (ai_adopt_2023["indic_is"] == "E_AI_TANY") &
    (ai_adopt_2023["unit"] == "PC_ENT") 
]

ai_adopt_2023 = ai_adopt_2023[ai_adopt_2023['geo'].isin(EU_EFTA)]
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023
3809         C  E_AI_TANY  PC_ENT  AT     12.31
3811         C  E_AI_TANY  PC_ENT  BE     15.31
3812         C  E_AI_TANY  PC_ENT  BG      2.55
3813         C  E_AI_TANY  PC_ENT  CY      3.81
3814         C  E_AI_TANY  PC_ENT  CZ      6.01
...        ...        ...     ...  ..       ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82
221157    S951  E_AI_TANY  PC_ENT  RO      0.00
221159    S951  E_AI_TANY  PC_ENT  SE      6.67
221160    S951  E_AI_TANY  PC_ENT  SI       NaN
221161    S951  E_AI_TANY  PC_ENT  SK      0.00

[1339 rows x 5 columns]


In [82]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s).strip().upper()
    match = re.fullmatch(r'([A-U])', s)

    if match:
        return match.group(1)
    else:
        return np.nan

ai_adopt_2023['nace_r2_1d'] = ai_adopt_2023['nace_r2'].map(nace_section_or_nan)
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023 nace_r2_1d
3809         C  E_AI_TANY  PC_ENT  AT     12.31          C
3811         C  E_AI_TANY  PC_ENT  BE     15.31          C
3812         C  E_AI_TANY  PC_ENT  BG      2.55          C
3813         C  E_AI_TANY  PC_ENT  CY      3.81          C
3814         C  E_AI_TANY  PC_ENT  CZ      6.01          C
...        ...        ...     ...  ..       ...        ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82        NaN
221157    S951  E_AI_TANY  PC_ENT  RO      0.00        NaN
221159    S951  E_AI_TANY  PC_ENT  SE      6.67        NaN
221160    S951  E_AI_TANY  PC_ENT  SI       NaN        NaN
221161    S951  E_AI_TANY  PC_ENT  SK      0.00        NaN

[1339 rows x 6 columns]


In [83]:
ai_adopt_2023.drop(columns=['nace_r2', 'indic_is', 'unit'], inplace=True) 
ai_adopt_2023.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(ai_adopt_2023)

       geo  num_2023 nace_r2
3809    AT     12.31       C
3811    BE     15.31       C
3812    BG      2.55       C
3813    CY      3.81       C
3814    CZ      6.01       C
...     ..       ...     ...
221156  PT      8.82     NaN
221157  RO      0.00     NaN
221159  SE      6.67     NaN
221160  SI       NaN     NaN
221161  SK      0.00     NaN

[1339 rows x 3 columns]


In [84]:
ai_adopt_2023 = ai_adopt_2023.dropna()
print(ai_adopt_2023)

       geo  num_2023 nace_r2
3809    AT     12.31       C
3811    BE     15.31       C
3812    BG      2.55       C
3813    CY      3.81       C
3814    CZ      6.01       C
...     ..       ...     ...
207613  PT     10.29       N
207614  RO      1.57       N
207616  SE      8.30       N
207617  SI      3.22       N
207618  SK     10.16       N

[242 rows x 3 columns]


In [85]:
ai_adopt_2023_agg = ai_adopt_2023.groupby(['geo', 'nace_r2'])['num_2023'].mean().reset_index()
ai_adopt_2023_agg.rename(columns={'num_2023': 'mn_ai_adopt'}, inplace=True)
print(ai_adopt_2023_agg)

    geo nace_r2  mn_ai_adopt
0    AT       C        12.31
1    AT       E         7.16
2    AT       F         4.28
3    AT       G         8.35
4    AT       H         8.31
..   ..     ...          ...
237  SK       H         3.32
238  SK       I         2.31
239  SK       J        21.62
240  SK       M        12.22
241  SK       N        10.16

[242 rows x 3 columns]


In [89]:
print(pd.unique(ai_adopt_2023_agg['nace_r2']))

['C' 'E' 'F' 'G' 'H' 'I' 'J' 'M' 'N']


In [86]:
c = len(pd.unique(ai_adopt_2023_agg['geo']))
print(f'Number of the available countries for the ai adoption variable {c}')

n = len(pd.unique(ai_adopt_2023_agg['nace_r2']))
print(f'Number of the available nace codes for the ai adoption variable {n}')

print(f'Maximum number of the available country-sector rows for the ai adoption variable {c*n}')
print(f'Real number of the available country-sector rows for the ai adoption variable {len(ai_adopt_2023_agg.index)}')


Number of the available countries for the ai adoption variable 28
Number of the available nace codes for the ai adoption variable 9
Maximum number of the available country-sector rows for the ai adoption variable 252
Real number of the available country-sector rows for the ai adoption variable 242


### Wages

The wage variable measures wages, salaries, bonuses, allowances, employer contributions to saving schemes, and remuneration in kind, as received by workers, in euros per hour. 


In [90]:
wages = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/lc.csv')
print(wages)

        freq currency     unit sizeclas nace_r2 lcstruct geo      num_2016  \
0          A      EUR  P_SAL_H    10-49       B      D01  AL  2.500000e+00   
1          A      EUR  P_SAL_H    10-49       B      D01  AT  3.099000e+01   
2          A      EUR  P_SAL_H    10-49       B      D01  BA  3.910000e+00   
3          A      EUR  P_SAL_H    10-49       B      D01  BE  3.548000e+01   
4          A      EUR  P_SAL_H    10-49       B      D01  BG           NaN   
...      ...      ...      ...      ...     ...      ...  ..           ...   
1254268    A      PPS    TOTAL    TOTAL     S96    D1111  RS  3.134016e+07   
1254269    A      PPS    TOTAL    TOTAL     S96    D1111  SI  4.127085e+07   
1254270    A      PPS    TOTAL    TOTAL     S96    D1111  SK  3.383858e+07   
1254271    A      PPS    TOTAL    TOTAL     S96    D1111  TR  4.904631e+08   
1254272    A      PPS    TOTAL    TOTAL     S96    D1111  UK  4.946674e+09   

        flag_2016      num_2020 flag_2020  
0             NaN  

In [91]:
# Filter the data 

wg_2016 = wages.copy()
wg_2016 = wg_2016[
    (wg_2016['currency'] == 'EUR') &
    (wg_2016['sizeclas'] == 'GE10') &  # 10 employees or more
    (wg_2016['unit'] == 'P_SAL_H') & # per employeein full-time equivalrnts, per hour
    (wg_2016['lcstruct'] == 'D111') # Wages and Salaries (excluding apprentices)
]

wg_2016 = wg_2016.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2020', 'flag_2020', 'flag_2016'])
wg_2016 = wg_2016[wg_2016['geo'].isin(EU_EFTA)]

print(wg_2016)

      nace_r2 geo  num_2016
57688     A01  SI       NaN
57718     A02  SI       NaN
57748     A03  SI       NaN
57806       B  AT     28.31
57808       B  BE     29.48
...       ...  ..       ...
73080     S96  PT      6.69
73081     S96  RO      1.97
73083     S96  SE     18.54
73084     S96  SI     10.67
73085     S96  SK      4.81

[3254 rows x 3 columns]


In [92]:
wg_2020 = wages.copy()

wg_2020 = wages.copy()
wg_2020 = wg_2020[
    (wg_2020['currency'] == 'EUR') &
    (wg_2020['sizeclas'] == 'GE10') &  # 10 employees or more
    (wg_2020['unit'] == 'P_SAL_H') & # per employeein full-time equivalrnts, per hour
    (wg_2020['lcstruct'] == 'D111') # Wages and Salaries (excluding apprentices)
]

wg_2020 = wg_2020.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2016', 'flag_2020', 'flag_2016'])
wg_2020 = wg_2020[wg_2020['geo'].isin(EU_EFTA)]

print(wg_2020)

      nace_r2 geo  num_2020
57688     A01  SI       NaN
57718     A02  SI       NaN
57748     A03  SI       NaN
57806       B  AT     28.62
57808       B  BE     31.33
...       ...  ..       ...
73080     S96  PT      8.09
73081     S96  RO      3.68
73083     S96  SE     19.97
73084     S96  SI     14.62
73085     S96  SK      6.96

[3254 rows x 3 columns]


In [93]:
wg_2016['nace_r2_1d'] = wg_2016['nace_r2'].map(nace_section_or_nan)

wg_2016.drop(columns=['nace_r2'], inplace=True) 
wg_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(wg_2016)

      geo  num_2016 nace_r2
57688  SI       NaN     NaN
57718  SI       NaN     NaN
57748  SI       NaN     NaN
57806  AT     28.31       B
57808  BE     29.48       B
...    ..       ...     ...
73080  PT      6.69     NaN
73081  RO      1.97     NaN
73083  SE     18.54     NaN
73084  SI     10.67     NaN
73085  SK      4.81     NaN

[3254 rows x 3 columns]


In [94]:
wg_2020['nace_r2_1d'] = wg_2020['nace_r2'].map(nace_section_or_nan)

wg_2020.drop(columns=['nace_r2'], inplace=True) 
wg_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(wg_2020)

      geo  num_2020 nace_r2
57688  SI       NaN     NaN
57718  SI       NaN     NaN
57748  SI       NaN     NaN
57806  AT     28.62       B
57808  BE     31.33       B
...    ..       ...     ...
73080  PT      8.09     NaN
73081  RO      3.68     NaN
73083  SE     19.97     NaN
73084  SI     14.62     NaN
73085  SK      6.96     NaN

[3254 rows x 3 columns]


In [95]:
wg_2016 = wg_2016.dropna()
wg_2016_agg = wg_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
wg_2016_agg.rename(columns={'num_2016': 'wg_2016'}, inplace=True)
print(wg_2016_agg)

    geo nace_r2  wg_2016
0    AT       B    28.31
1    AT       C    26.85
2    AT       D    39.23
3    AT       E    21.64
4    AT       F    24.10
..   ..     ...      ...
518  SK       O     7.56
519  SK       P     7.22
520  SK       Q     7.51
521  SK       R     5.70
522  SK       S     5.05

[523 rows x 3 columns]


In [96]:
wg_2020 = wg_2020.dropna()
wg_2020_agg = wg_2020.groupby(['geo', 'nace_r2'])['num_2020'].mean().reset_index()
wg_2020_agg.rename(columns={'num_2020': 'wg_2020'}, inplace=True)
print(wg_2020_agg)

    geo nace_r2  wg_2020
0    AT       B    28.62
1    AT       C    30.07
2    AT       D    43.39
3    AT       E    24.50
4    AT       F    25.83
..   ..     ...      ...
522  SK       O    10.59
523  SK       P    10.75
524  SK       Q    10.54
525  SK       R     9.09
526  SK       S     6.81

[527 rows x 3 columns]


In [97]:
print(pd.unique(wg_2016_agg['nace_r2']))
print(pd.unique(wg_2020_agg['nace_r2']))

['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'P' 'Q' 'R' 'S' 'O']
['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'P' 'Q' 'R' 'S' 'O']


In [98]:
print('WAGES 2016')
c_1 = len(pd.unique(wg_2016_agg['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(wg_2016_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(wg_2016_agg.index)}')

print('WAGES 2020')
c_2 = len(pd.unique(wg_2020_agg['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(wg_2020_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(wg_2020_agg.index)}')

WAGES 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 18
Maximum number of the available country-sector rows for the wages in 2016 variable 540
Real number of the available country-sector rows for the wages in 2016 variable 523
WAGES 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 18
Maximum number of the available country-sector rows for the wages in 2020 variable 540
Real number of the available country-sector rows for the wages in 2020 variable 527


### Labour cost

Labor costs measure costs to firms, which include wage and non-wage costs minus subsidies, in euros per hour.

In [99]:
# Filter the data 

lc_2016 = wages.copy()

lc_2016 = lc_2016[
    (lc_2016['currency'] == 'EUR') &
    (lc_2016['sizeclas'] == 'GE10') &  # 10 employees or more
    (lc_2016['unit'] == 'P_SAL_H') & # per employeein full-time equivalents, per hour
    (lc_2016['lcstruct'] == 'D01') # Total labour costs (excluding apprentices)
]

lc_2016 = lc_2016.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2020', 'flag_2020', 'flag_2016'])
lc_2016 = lc_2016[lc_2016['geo'].isin(EU_EFTA)]

print(lc_2016)

      nace_r2 geo  num_2016
57678     A01  SI       NaN
57708     A02  SI       NaN
57738     A03  SI       NaN
57760       B  AT     39.61
57762       B  BE     40.70
...       ...  ..       ...
73034     S96  PT      8.60
73035     S96  RO      2.46
73037     S96  SE     26.03
73038     S96  SI     12.39
73039     S96  SK      6.44

[3254 rows x 3 columns]


In [100]:
lc_2020 = wages.copy()

lc_2020 = lc_2020[
    (lc_2020['currency'] == 'EUR') &
    (lc_2020['sizeclas'] == 'GE10') &  # 10 employees or more
    (lc_2020['unit'] == 'P_SAL_H') & # per employeein full-time equivalents, per hour
    (lc_2020['lcstruct'] == 'D01') # Total labour costs (excluding apprentices)
]

lc_2020 = lc_2020.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2016', 'flag_2020', 'flag_2016'])
lc_2020 = lc_2020[lc_2020['geo'].isin(EU_EFTA)]

print(lc_2020)

      nace_r2 geo  num_2020
57678     A01  SI       NaN
57708     A02  SI       NaN
57738     A03  SI       NaN
57760       B  AT     41.15
57762       B  BE     42.47
...       ...  ..       ...
73034     S96  PT      9.38
73035     S96  RO      3.85
73037     S96  SE     27.19
73038     S96  SI     15.83
73039     S96  SK      8.90

[3254 rows x 3 columns]


In [101]:
lc_2016['nace_r2_1d'] = lc_2016['nace_r2'].map(nace_section_or_nan)

lc_2016.drop(columns=['nace_r2'], inplace=True) 
lc_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(lc_2016)

      geo  num_2016 nace_r2
57678  SI       NaN     NaN
57708  SI       NaN     NaN
57738  SI       NaN     NaN
57760  AT     39.61       B
57762  BE     40.70       B
...    ..       ...     ...
73034  PT      8.60     NaN
73035  RO      2.46     NaN
73037  SE     26.03     NaN
73038  SI     12.39     NaN
73039  SK      6.44     NaN

[3254 rows x 3 columns]


In [102]:
lc_2020['nace_r2_1d'] = lc_2020['nace_r2'].map(nace_section_or_nan)

lc_2020.drop(columns=['nace_r2'], inplace=True) 
lc_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(lc_2020)

      geo  num_2020 nace_r2
57678  SI       NaN     NaN
57708  SI       NaN     NaN
57738  SI       NaN     NaN
57760  AT     41.15       B
57762  BE     42.47       B
...    ..       ...     ...
73034  PT      9.38     NaN
73035  RO      3.85     NaN
73037  SE     27.19     NaN
73038  SI     15.83     NaN
73039  SK      8.90     NaN

[3254 rows x 3 columns]


In [103]:
lc_2016 = lc_2016.dropna()
lc_2016_agg = lc_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
lc_2016_agg.rename(columns={'num_2016': 'lc_2016'}, inplace=True)
print(lc_2016_agg)

    geo nace_r2  lc_2016
0    AT       B    39.61
1    AT       C    36.47
2    AT       D    54.17
3    AT       E    29.26
4    AT       F    34.40
..   ..     ...      ...
518  SK       O    10.36
519  SK       P     9.82
520  SK       Q    10.15
521  SK       R     7.65
522  SK       S     6.78

[523 rows x 3 columns]


In [104]:
lc_2020 = lc_2020.dropna()
lc_2020_agg = lc_2020.groupby(['geo', 'nace_r2'])['num_2020'].mean().reset_index()
lc_2020_agg.rename(columns={'num_2020': 'lc_2020'}, inplace=True)
print(lc_2020_agg)

    geo nace_r2  lc_2020
0    AT       B    41.15
1    AT       C    40.84
2    AT       D    59.64
3    AT       E    33.50
4    AT       F    37.80
..   ..     ...      ...
522  SK       O    14.43
523  SK       P    14.65
524  SK       Q    14.24
525  SK       R    11.94
526  SK       S     9.04

[527 rows x 3 columns]


In [105]:
print(pd.unique(lc_2016_agg['nace_r2']))
print(pd.unique(lc_2020_agg['nace_r2']))

['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'P' 'Q' 'R' 'S' 'O']
['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'P' 'Q' 'R' 'S' 'O']


In [106]:
print('LABOUR COST 2016')
c_1 = len(pd.unique(lc_2016_agg['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(lc_2016_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(lc_2016_agg.index)}')

print('LABOUR COST 2020')
c_2 = len(pd.unique(lc_2020_agg['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(lc_2020_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(lc_2020_agg.index)}')

LABOUR COST 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 18
Maximum number of the available country-sector rows for the wages in 2016 variable 540
Real number of the available country-sector rows for the wages in 2016 variable 523
LABOUR COST 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 18
Maximum number of the available country-sector rows for the wages in 2020 variable 540
Real number of the available country-sector rows for the wages in 2020 variable 527


### Cost of capital 

Cost of capital is the harmonized bank lending rate on outstanding loans with more than 5 years' maturity; it varies across countries only. 

In [69]:
cap_cst = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/interest_rates_geo.csv')
print(cap_cst)

     year geo  inter_rt_mean
0    2016  BE       2.650833
1    2017  BE       2.320833
2    2018  BE       2.133333
3    2019  BE       2.010833
4    2020  BE       1.824167
..    ...  ..            ...
185  2021  FI       1.168333
186  2022  FI       1.452500
187  2023  FI       3.832500
188  2024  FI       4.350833
189  2025  FI       3.437500

[190 rows x 3 columns]


In [ ]:
cap_cst_2016 = cap_cst.copy()
cap_cst_2016 = cap_cst_2016[
    (cap_cst_2016['year'] == 2016) 
]
cap_cst_2016.rename(columns={'inter_rt_mean' : 'inter_rt_2016'}, inplace=True)

cap_cst_2016 = cap_cst_2016.drop(columns=['year'])
print(cap_cst_2016)

In [ ]:
cap_cst_2020 = cap_cst.copy()
cap_cst_2020 = cap_cst_2020[
    (cap_cst_2020['year'] == 2020) 
]
cap_cst_2020.rename(columns={'inter_rt_mean' : 'inter_rt_2020'}, inplace=True)

cap_cst_2020 = cap_cst_2020.drop(columns=['year'])
print(cap_cst_2020)

### Concentration 

Concentrationis the share of firms that have more than 250 employees. 

In [74]:
conctr = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/conctr.csv')
print(conctr)

       freq nace_r2 indic_sb size_emp geo  num_2016 flag_2016  num_2020  \
0         A       B   V11110      0-9  AL       NaN       NaN     510.0   
1         A       B   V11110      0-9  AT     226.0       NaN     197.0   
2         A       B   V11110      0-9  BA     129.0         e      98.0   
3         A       B   V11110      0-9  BE     144.0       NaN     141.0   
4         A       B   V11110      0-9  BG     224.0       NaN     199.0   
...     ...     ...      ...      ...  ..       ...       ...       ...   
483382    A     S95   V92100    TOTAL  RS       4.2         p       4.7   
483383    A     S95   V92100    TOTAL  SE       1.2       NaN       1.0   
483384    A     S95   V92100    TOTAL  SI       1.5       NaN       1.4   
483385    A     S95   V92100    TOTAL  SK       1.4       NaN       1.4   
483386    A     S95   V92100    TOTAL  UK       7.3         b       NaN   

       flag_2020  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4   

In [78]:
conctr_2016 = conctr.copy()

conctr_2016 = conctr_2016[
    (conctr_2016['indic_sb'] == 'V11110') # Enterprices - number
]

conctr_2016 = conctr_2016.drop(columns=['freq','indic_sb', 'num_2020', 'flag_2020', 'flag_2016'])
conctr_2016 = conctr_2016[conctr_2016['geo'].isin(EU_EFTA)]

print(conctr_2016)

conctr_2020 = conctr.copy()

conctr_2020 = conctr_2020[
    (conctr_2020['indic_sb'] == 'V11110')  # Enterprices - number
]

conctr_2020 = conctr_2020.drop(columns=['freq', 'indic_sb', 'num_2016', 'flag_2020', 'flag_2016'])
conctr_2020 = conctr_2020[conctr_2020['geo'].isin(EU_EFTA)]

print(conctr_2020)

       nace_r2 size_emp geo  num_2016
1            B      0-9  AT     226.0
3            B      0-9  BE     144.0
4            B      0-9  BG     224.0
5            B      0-9  CH      85.0
6            B      0-9  CY       NaN
...        ...      ...  ..       ...
481946     S95    TOTAL  PT    4864.0
481947     S95    TOTAL  RO    3832.0
481949     S95    TOTAL  SE    4335.0
481950     S95    TOTAL  SI    1188.0
481951     S95    TOTAL  SK    2922.0

[36663 rows x 4 columns]
       nace_r2 size_emp geo  num_2020
1            B      0-9  AT     197.0
3            B      0-9  BE     141.0
4            B      0-9  BG     199.0
5            B      0-9  CH      85.0
6            B      0-9  CY       NaN
...        ...      ...  ..       ...
481946     S95    TOTAL  PT    4690.0
481947     S95    TOTAL  RO    4005.0
481949     S95    TOTAL  SE    4027.0
481950     S95    TOTAL  SI    1243.0
481951     S95    TOTAL  SK    3034.0

[36663 rows x 4 columns]


In [107]:
conctr_2016['nace_r2_1d'] = conctr_2016['nace_r2'].map(nace_section_or_nan)

conctr_2016.drop(columns=['nace_r2'], inplace=True) 
conctr_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(conctr_2016)

       size_emp geo  num_2016 nace_r2
1           0-9  AT     226.0       B
3           0-9  BE     144.0       B
4           0-9  BG     224.0       B
5           0-9  CH      85.0       B
6           0-9  CY       NaN       B
...         ...  ..       ...     ...
481946    TOTAL  PT    4864.0     NaN
481947    TOTAL  RO    3832.0     NaN
481949    TOTAL  SE    4335.0     NaN
481950    TOTAL  SI    1188.0     NaN
481951    TOTAL  SK    2922.0     NaN

[36663 rows x 4 columns]


In [111]:
conctr_2020['nace_r2_1d'] = conctr_2020['nace_r2'].map(nace_section_or_nan)

conctr_2020.drop(columns=['nace_r2'], inplace=True) 
conctr_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(conctr_2020)

       size_emp geo  num_2020 nace_r2
1           0-9  AT     197.0       B
3           0-9  BE     141.0       B
4           0-9  BG     199.0       B
5           0-9  CH      85.0       B
6           0-9  CY       NaN       B
...         ...  ..       ...     ...
481946    TOTAL  PT    4690.0     NaN
481947    TOTAL  RO    4005.0     NaN
481949    TOTAL  SE    4027.0     NaN
481950    TOTAL  SI    1243.0     NaN
481951    TOTAL  SK    3034.0     NaN

[36663 rows x 4 columns]


In [ ]:
conctr_2016 = conctr_2016.dropna()
print(conctr_2016)

       size_emp geo  num_2016 nace_r2
1           0-9  AT     226.0       B
3           0-9  BE     144.0       B
4           0-9  BG     224.0       B
5           0-9  CH      85.0       B
7           0-9  CZ     281.0       B
...         ...  ..       ...     ...
469726    TOTAL  PT  163936.0       N
469727    TOTAL  RO   20802.0       N
469729    TOTAL  SE   39120.0       N
469730    TOTAL  SI    7348.0       N
469731    TOTAL  SK   30296.0       N

[2091 rows x 4 columns]


In [114]:
conctr_2020 = conctr_2020.dropna()
print(conctr_2020)

       size_emp geo  num_2020 nace_r2
1           0-9  AT     197.0       B
3           0-9  BE     141.0       B
4           0-9  BG     199.0       B
5           0-9  CH      85.0       B
7           0-9  CZ     331.0       B
...         ...  ..       ...     ...
469726    TOTAL  PT  176636.0       N
469727    TOTAL  RO   25197.0       N
469729    TOTAL  SE   39052.0       N
469730    TOTAL  SI    8426.0       N
469731    TOTAL  SK   43632.0       N

[2094 rows x 4 columns]


In [116]:
conctr_2016.to_excel('test.xlsx')

In [ ]:
# Filter to only the needed size_emp values
conctr_2016_ft = conctr_2016[conctr_2016['size_emp'].isin(['GE250', 'TOTAL'])]

pivoted = conctr_2016_ft.pivot_table(
    index=['geo', 'nace_r2'],
    columns='size_emp',
    values='num_2016'
).reset_index()

pivoted.columns.name = None
pivoted = pivoted.rename(columns={'GE250': 'ge250', 'TOTAL': 'total'})

pivoted['conctr_2016'] = (pivoted['ge250'] / pivoted['total']) * 100

print(pivoted.head())

  geo nace_r2  ge250    total  conctr_2016
0  AT       B    5.0    348.0     1.436782
1  AT       C  471.0  25037.0     1.881216
2  AT       D   22.0   2430.0     0.905350
3  AT       E   11.0   2170.0     0.506912
4  AT       F   69.0  35078.0     0.196704


In [119]:
conctr_2016_rate = pivoted[['geo', 'nace_r2', 'conctr_2016']]
print(conctr_2016_rate)

    geo nace_r2  conctr_2016
0    AT       B     1.436782
1    AT       C     1.881216
2    AT       D     0.905350
3    AT       E     0.506912
4    AT       F     0.196704
..   ..     ...          ...
355  SK       I          NaN
356  SK       J     0.128777
357  SK       L          NaN
358  SK       M     0.028388
359  SK       N     0.155136

[360 rows x 3 columns]


In [122]:
conctr_2016_rate = conctr_2016_rate.dropna()
print(conctr_2016_rate)

    geo nace_r2  conctr_2016
0    AT       B     1.436782
1    AT       C     1.881216
2    AT       D     0.905350
3    AT       E     0.506912
4    AT       F     0.196704
..   ..     ...          ...
353  SK       G     0.075039
354  SK       H     0.252366
356  SK       J     0.128777
358  SK       M     0.028388
359  SK       N     0.155136

[341 rows x 3 columns]


In [123]:
# Filter to only the needed size_emp values
conctr_2020_ft = conctr_2020[conctr_2020['size_emp'].isin(['GE250', 'TOTAL'])]

pivoted = conctr_2020_ft.pivot_table(
    index=['geo', 'nace_r2'],
    columns='size_emp',
    values='num_2020'
).reset_index()

pivoted.columns.name = None
pivoted = pivoted.rename(columns={'GE250': 'ge250', 'TOTAL': 'total'})

pivoted['conctr_2020'] = (pivoted['ge250'] / pivoted['total']) * 100

print(pivoted.head())

  geo nace_r2  ge250    total  conctr_2020
0  AT       B    4.0    307.0     1.302932
1  AT       C  488.0  25727.0     1.896840
2  AT       D   23.0   2429.0     0.946892
3  AT       E   11.0   2126.0     0.517404
4  AT       F   85.0  37261.0     0.228121


In [124]:
conctr_2020_rate = pivoted[['geo', 'nace_r2', 'conctr_2020']]
conctr_2020_rate = conctr_2020_rate.dropna()
print(conctr_2020_rate)

    geo nace_r2  conctr_2020
0    AT       B     1.302932
1    AT       C     1.896840
2    AT       D     0.946892
3    AT       E     0.517404
4    AT       F     0.228121
..   ..     ...          ...
353  SK       G     0.092376
354  SK       H     0.240080
356  SK       J     0.122337
358  SK       M     0.024040
359  SK       N     0.073341

[341 rows x 3 columns]


In [125]:
print(pd.unique(conctr_2016_rate['nace_r2']))
print(pd.unique(conctr_2020_rate['nace_r2']))

['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'L' 'M' 'N']
['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'L' 'M' 'N']


In [126]:
print('CONCENTRATION RATE 2016')
c_1 = len(pd.unique(conctr_2016_rate['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(conctr_2016_rate['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(conctr_2016_rate.index)}')

print('CONCENTRATION RATE 2020')
c_2 = len(pd.unique(conctr_2020_rate['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(conctr_2020_rate['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(conctr_2020_rate.index)}')

CONCENTRATION RATE 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 12
Maximum number of the available country-sector rows for the wages in 2016 variable 360
Real number of the available country-sector rows for the wages in 2016 variable 341
CONCENTRATION RATE 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 12
Maximum number of the available country-sector rows for the wages in 2020 variable 360
Real number of the available country-sector rows for the wages in 2020 variable 341


### Computer use 

Computer use is the proportion of firms using computers. Data available only for 2016 year.

In [151]:
comp = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/comp.csv')
print(comp)

     freq size_emp nace_r2 indic_is         unit geo\TIME_PERIOD  num_2016  \
0       A     GE10       C   P_CUSE       PC_EMP              AT       NaN   
1       A     GE10       C   P_CUSE       PC_EMP              BA       NaN   
2       A     GE10       C   P_CUSE       PC_EMP              BE     57.93   
3       A     GE10       C   P_CUSE       PC_EMP              BG     19.17   
4       A     GE10       C   P_CUSE       PC_EMP              CY     32.96   
...   ...      ...     ...      ...          ...             ...       ...   
8815    A     GE10    S951   P_IUSE  PC_EMP_IACC              SE       NaN   
8816    A     GE10    S951   P_IUSE  PC_EMP_IACC              SI       NaN   
8817    A     GE10    S951   P_IUSE  PC_EMP_IACC              SK       NaN   
8818    A     GE10    S951   P_IUSE  PC_EMP_IACC              TR       NaN   
8819    A     GE10    S951   P_IUSE  PC_EMP_IACC              UK       NaN   

     flag_2016  num_2020 flag_2020  
0          NaN       NaN  

In [152]:
# Filter the data 
# size_emp = GE10 (only that parameter available)
comp_2016 = comp.copy()

comp_2016 = comp_2016[
    (comp_2016['indic_is'] == 'P_CUSE') & # Person employed using computer
    (comp_2016['unit'] == 'PC_EMP') # Percentage of total employment 
]

comp_2016 = comp_2016.drop(columns=['freq', 'size_emp', 'indic_is', 'unit', 'num_2020', 'flag_2020', 'flag_2016'])
comp_2016 = comp_2016.rename(columns={'geo\\TIME_PERIOD': 'geo'})
comp_2016 = comp_2016[comp_2016['geo'].isin(EU_EFTA)]

print(comp_2016)

     nace_r2 geo  num_2016
0          C  AT       NaN
2          C  BE     57.93
3          C  BG     19.17
4          C  CY     32.96
5          C  CZ     46.41
...      ...  ..       ...
8622    S951  PT     90.71
8623    S951  RO     72.09
8625    S951  SE       NaN
8626    S951  SI       NaN
8627    S951  SK     98.37

[1037 rows x 3 columns]


In [153]:
comp_2016['nace_r2_1d'] = comp_2016['nace_r2'].map(nace_section_or_nan)

comp_2016.drop(columns=['nace_r2'], inplace=True) 
comp_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(comp_2016)

     geo  num_2016 nace_r2
0     AT       NaN       C
2     BE     57.93       C
3     BG     19.17       C
4     CY     32.96       C
5     CZ     46.41       C
...   ..       ...     ...
8622  PT     90.71     NaN
8623  RO     72.09     NaN
8625  SE       NaN     NaN
8626  SI       NaN     NaN
8627  SK     98.37     NaN

[1037 rows x 3 columns]


In [154]:
comp_2016 = comp_2016.dropna()
comp_2016_agg = comp_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
comp_2016_agg.rename(columns={'num_2016': 'comp_2016'}, inplace=True)
print(comp_2016_agg)

    geo nace_r2  comp_2016
0    BE       C      57.93
1    BE       F      41.86
2    BE       G      67.43
3    BE       I      36.91
4    BE       J      97.64
..   ..     ...        ...
139  SK       G      63.52
140  SK       H      48.62
141  SK       I      30.96
142  SK       J      96.90
143  SK       N      33.21

[144 rows x 3 columns]


### Energy cost 

Energy cost is the price of electricity in euros per kilowatt-hour for firms that use between 2 and 20 thousand megawatt-hours per year, including all taxes and fees; it varies across countries only.


In [160]:
enrg_cst = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/enrg_cst.csv')
print(enrg_cst)

     freq  product   nrg_cons unit    tax currency geo     2016     2020  \
0       S     6000  MWH20-499  KWH  I_TAX      EUR  AL      NaN  0.12440   
1       S     6000  MWH20-499  KWH  I_TAX      EUR  AT  0.15045  0.16190   
2       S     6000  MWH20-499  KWH  I_TAX      EUR  BA  0.10040  0.10610   
3       S     6000  MWH20-499  KWH  I_TAX      EUR  BE  0.18490  0.18465   
4       S     6000  MWH20-499  KWH  I_TAX      EUR  BG  0.12105  0.11335   
...   ...      ...        ...  ...    ...      ...  ..      ...      ...   
2881    S     6000    TOT_KWH  KWH  X_VAT      PPS  RS      NaN      NaN   
2882    S     6000    TOT_KWH  KWH  X_VAT      PPS  SE      NaN  0.04425   
2883    S     6000    TOT_KWH  KWH  X_VAT      PPS  SI      NaN      NaN   
2884    S     6000    TOT_KWH  KWH  X_VAT      PPS  SK      NaN      NaN   
2885    S     6000    TOT_KWH  KWH  X_VAT      PPS  TR      NaN      NaN   

     flag_2016 flag_2020  
0          NaN         e  
1          NaN       NaN  
2     

In [163]:
# Filter the data 
# product [6000] - Electrical energy
# unit [KWH] - Kilowatt-hour 

# 2016
enrg_cst_2016 = enrg_cst.copy()

enrg_cst_2016 = enrg_cst_2016[
    (enrg_cst_2016['nrg_cons'] == 'MWH2000-19999') & # Consumption from 2 000 MWh to 19 999 MWh - band ID
    (enrg_cst_2016['tax'] == 'I_TAX') & # all taxes and levies included
    (enrg_cst_2016['currency'] == 'EUR') 
]

enrg_cst_2016 = enrg_cst_2016.drop(columns=['freq', 'product', 'unit', 'currency', 'nrg_cons', 'tax', '2020', 'flag_2020', 'flag_2016'])
enrg_cst_2016 = enrg_cst_2016[enrg_cst_2016['geo'].isin(EU_EFTA)]
enrg_cst_2016 = enrg_cst_2016.dropna()

print(enrg_cst_2016)

# 2020
enrg_cst_2020 = enrg_cst.copy()

enrg_cst_2020 = enrg_cst_2020[
    (enrg_cst_2020['nrg_cons'] == 'MWH2000-19999') & # Consumption from 2 000 MWh to 19 999 MWh - band ID
    (enrg_cst_2020['tax'] == 'I_TAX') & # all taxes and levies included
    (enrg_cst_2020['currency'] == 'EUR') 
]

enrg_cst_2020 = enrg_cst_2020.drop(columns=['freq', 'product', 'unit', 'currency', 'nrg_cons', 'tax', '2016', 'flag_2020', 'flag_2016'])
enrg_cst_2020 = enrg_cst_2020[enrg_cst_2020['geo'].isin(EU_EFTA)]
enrg_cst_2020 = enrg_cst_2020.dropna()

print(enrg_cst_2020)

    geo     2016
370  AT  0.10185
372  BE  0.11605
373  BG  0.08790
374  CY  0.12515
375  CZ  0.07670
376  DE  0.16985
377  DK  0.26470
379  EE  0.09685
380  EL  0.10125
381  ES  0.10685
383  FI  0.08170
384  FR  0.09205
386  HR  0.09650
387  HU  0.09420
388  IE  0.11275
389  IS  0.06190
390  IT  0.15915
392  LT  0.09845
393  LU  0.05850
394  LV  0.12985
398  MT  0.12880
399  NL  0.09320
400  NO  0.08100
401  PL  0.08775
402  PT  0.12585
403  RO  0.08420
405  SE  0.06805
406  SI  0.08710
407  SK  0.11790
    geo     2020
370  AT  0.11970
372  BE  0.11925
373  BG  0.09425
374  CY  0.15860
375  CZ  0.09860
376  DE  0.19110
377  DK  0.23230
379  EE  0.09035
380  EL  0.09740
381  ES  0.10165
383  FI  0.08510
384  FR  0.09900
386  HR  0.10670
387  HU  0.11085
388  IE  0.11545
389  IS  0.05450
390  IT  0.14530
392  LT  0.10260
393  LU  0.07125
394  LV  0.10485
398  MT  0.12425
399  NL  0.11855
400  NO  0.04910
401  PL  0.11950
402  PT  0.12405
403  RO  0.11610
405  SE  0.06445
406  SI  0.104

### Human Capital 

Human capital is the proportion of employees that have a tertiary education. 

In [176]:
hum_cap = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/human_capital.csv', index_col = 0)
print(hum_cap)

       freq unit nace_r2 isced11     age sex geo\TIME_PERIOD  num_2016  \
0         A   PC       A   ED0-2  Y15-24   F              AT       NaN   
1         A   PC       A   ED0-2  Y15-24   F              BA       NaN   
2         A   PC       A   ED0-2  Y15-24   F              BE       NaN   
3         A   PC       A   ED0-2  Y15-24   F              BG       NaN   
4         A   PC       A   ED0-2  Y15-24   F              CH      33.1   
...     ...  ...     ...     ...     ...  ..             ...       ...   
241725    A   PC       U   ED5-8  Y55-74   T              SE       NaN   
241726    A   PC       U   ED5-8  Y55-74   T              SI       NaN   
241727    A   PC       U   ED5-8  Y55-74   T              SK       NaN   
241728    A   PC       U   ED5-8  Y55-74   T              TR       NaN   
241729    A   PC       U   ED5-8  Y55-74   T              UK       NaN   

       flag_2016  num_2020 flag_2020  
0              u       NaN         u  
1            NaN       NaN       

In [177]:
# Filter the data
# unit [PC] - Percentage

# 2016
hum_cap = hum_cap.rename(columns={'geo\\TIME_PERIOD': 'geo'})
hum_cap_2016 = hum_cap.copy()

hum_cap_2016 = hum_cap_2016[
    (hum_cap_2016['isced11'] == 'ED5-8') & # Tertiary education (levels 5-8)
    (hum_cap_2016['sex'] == 'T') & 
    (hum_cap_2016['age'] == 'Y25-64') 
]

hum_cap_2016 = hum_cap_2016.drop(columns=['freq', 'unit', 'sex', 'age', 'isced11', 'num_2020', 'flag_2020', 'flag_2016'])
hum_cap_2016 = hum_cap_2016[hum_cap_2016['geo'].isin(EU_EFTA)]
hum_cap_2016 = hum_cap_2016.dropna()

print(hum_cap_2016)

# 2020
hum_cap_2020 = hum_cap.copy()

hum_cap_2020 = hum_cap_2020[
    (hum_cap_2020['isced11'] == 'ED5-8') & # Tertiary education (levels 5-8)
    (hum_cap_2020['sex'] == 'T') & 
    (hum_cap_2020['age'] == 'Y25-64') 
]

hum_cap_2020 = hum_cap_2020.drop(columns=['freq', 'unit', 'sex', 'age', 'isced11', 'num_2016', 'flag_2020', 'flag_2016'])
hum_cap_2020 = hum_cap_2020[hum_cap_2020['geo'].isin(EU_EFTA)]
hum_cap_2020 = hum_cap_2020.dropna()

print(hum_cap_2020)

       nace_r2 geo  num_2016
10234        A  AT      22.5
10236        A  BE      18.0
10237        A  BG       9.3
10238        A  CH      36.9
10239        A  CY      12.7
...        ...  ..       ...
241245       U  DE      49.4
241253       U  FR      75.3
241258       U  IT      46.1
241260       U  LU      89.7
241264       U  MT      56.2

[593 rows x 3 columns]
       nace_r2 geo  num_2020
10234        A  AT      31.5
10237        A  BG       8.6
10238        A  CH      27.3
10239        A  CY      17.7
10240        A  CZ      11.4
...        ...  ..       ...
241246       U  DK      84.2
241249       U  EL      57.6
241253       U  FR      80.8
241258       U  IT      54.5
241260       U  LU      89.6

[594 rows x 3 columns]


In [178]:
hum_cap_2016['nace_r2_1d'] = hum_cap_2016['nace_r2'].map(nace_section_or_nan)

hum_cap_2016.drop(columns=['nace_r2'], inplace=True) 
hum_cap_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(hum_cap_2016)

hum_cap_2020['nace_r2_1d'] = hum_cap_2020['nace_r2'].map(nace_section_or_nan)

hum_cap_2020.drop(columns=['nace_r2'], inplace=True) 
hum_cap_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(hum_cap_2020)

       geo  num_2016 nace_r2
10234   AT      22.5       A
10236   BE      18.0       A
10237   BG       9.3       A
10238   CH      36.9       A
10239   CY      12.7       A
...     ..       ...     ...
241245  DE      49.4       U
241253  FR      75.3       U
241258  IT      46.1       U
241260  LU      89.7       U
241264  MT      56.2       U

[593 rows x 3 columns]
       geo  num_2020 nace_r2
10234   AT      31.5       A
10237   BG       8.6       A
10238   CH      27.3       A
10239   CY      17.7       A
10240   CZ      11.4       A
...     ..       ...     ...
241246  DK      84.2       U
241249  EL      57.6       U
241253  FR      80.8       U
241258  IT      54.5       U
241260  LU      89.6       U

[594 rows x 3 columns]


In [179]:
hum_cap_2016_agg = hum_cap_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
hum_cap_2016_agg.rename(columns={'num_2016': 'hum_cap_2016'}, inplace=True)
print(hum_cap_2016_agg)

hum_cap_2020_agg = hum_cap_2020.groupby(['geo', 'nace_r2'])['num_2020'].mean().reset_index()
hum_cap_2020_agg.rename(columns={'num_2020': 'hum_cap_2020'}, inplace=True)
print(hum_cap_2020_agg)

    geo nace_r2  hum_cap_2016
0    AT       A          22.5
1    AT       C          30.9
2    AT       D          48.0
3    AT       E          20.9
4    AT       F          23.0
..   ..     ...           ...
548  SK       O          35.2
549  SK       P          61.7
550  SK       Q          31.4
551  SK       R          23.3
552  SK       S          19.7

[553 rows x 3 columns]
    geo nace_r2  hum_cap_2020
0    AT       A          31.5
1    AT       C          31.8
2    AT       D          48.2
3    AT       E          19.7
4    AT       F          22.8
..   ..     ...           ...
550  SK       O          45.2
551  SK       P          65.2
552  SK       Q          36.3
553  SK       R          31.3
554  SK       S          26.0

[555 rows x 3 columns]


In [180]:
print(pd.unique(hum_cap_2016_agg['nace_r2']))
print(pd.unique(hum_cap_2020_agg['nace_r2']))

['A' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'U' 'T' 'B']
['A' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S'
 'U' 'B' 'T']


In [182]:
print('HUMAN CAPITAL RATIO 2016')
c_1 = len(pd.unique(hum_cap_2016_agg['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(hum_cap_2016_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(hum_cap_2016_agg.index)}')

print('HUMAN CAPITAL RATIO 2020')
c_2 = len(pd.unique(hum_cap_2020_agg['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(hum_cap_2020_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(hum_cap_2020_agg.index)}')

HUMAN CAPITAL RATIO 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 21
Maximum number of the available country-sector rows for the wages in 2016 variable 630
Real number of the available country-sector rows for the wages in 2016 variable 553
HUMAN CAPITAL RATIO 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 21
Maximum number of the available country-sector rows for the wages in 2020 variable 630
Real number of the available country-sector rows for the wages in 2020 variable 555
